# Detección de agua e inundación en Lima Sur con Sentinel-1 y U-Net

Pipeline completo del informe (datasets completos, ya no la muestra):

| Ruta | Datos | Uso |
|---|---|---|
| **A** | Sen1Floods11 Colombia, Kuro Siwo (bloque 00), UrbanSARFloods | entrenamiento y validación (partición por eventos) |
| **B** | Sentinel-1 Lima 2017 + EMSN038; Lima Sur 2023 + CONIDA | prueba y aplicación, **sin** reentrenar |
| extra | Sen1Floods11 Bolivia (15 parches manuales) | comparación con la literatura |

Se entrenan dos configuraciones:

* **`agua`** (la del informe): 2 canales (VV, VH durante el evento) → presencia de agua.
* **`cambio`**: 4 canales (VV, VH antes + VV, VH durante) → inundación nueva. Kuro Siwo y UrbanSARFloods traen imagen previa, y Bountos et al. (2024) muestran que apilar fechas es lo que mejor funciona. Además ataca el problema principal de Lima: la arena del desierto es oscura en radar **en ambas fechas**, así que no “cambia”.

> **Qué responde y qué no:** el modelo detecta *dónde hubo agua en superficie en el momento en que pasó el satélite*. No predice lluvia ni pronostica inundaciones futuras.

Todo se configura en `src/config.py`. Con la variable de entorno `PML_RAPIDO=1` el notebook corre de punta a punta en pocos minutos (prueba de humo).

## 0. Entorno

In [ ]:
import sys, json, warnings
from pathlib import Path
RAIZ = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore", message=".*Measured \\(M\\) geometry.*")

import numpy as np, pandas as pd, torch, rasterio, geopandas as gpd
import matplotlib.pyplot as plt
from src import config as C
from src import datos as D, entrenamiento as E, inferencia as I

C.SALIDA.mkdir(parents=True, exist_ok=True)
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
REHACER_PARCHES = False   # True para volver a generar inventario y parches
REENTRENAR = False        # True para reentrenar aunque ya exista el .pt
print(f"datos: {C.DATOS}\nsalida: {C.SALIDA}\ndispositivo: {DISPOSITIVO}"
      + (f" ({torch.cuda.get_device_name(0)})" if DISPOSITIVO == "cuda" else ""))

## 1. Inventario de fuentes (Ruta A)
Un registro por archivo / tesela. La partición se asigna por **evento completo** antes de muestrear y de calcular la normalización.

In [ ]:
items = D.listar_items()
items.groupby(["particion", "dataset", "evento"]).agg(items=("item", "size"), con_imagen_previa=("tiene_pre", "sum"))

## 2. Inventario de parches y muestreo por evento
Se recorren solo las etiquetas para conocer la fracción válida y de agua de cada ventana de 256×256. Después se aplica un tope de `MAX_PARCHES_EVENTO` parches por evento con `FRAC_CON_AGUA` de parches con agua, para que Harvey (5 001 teselas) y los chips sin inundación de UrbanSARFloods no dominen.

In [ ]:
ruta_inv = C.SALIDA / "inventario.csv"
if ruta_inv.exists() and not REHACER_PARCHES:
    inv = pd.read_csv(ruta_inv)
else:
    inv = D.inventario(items)
    inv.to_csv(ruta_inv, index=False)
seleccion = D.muestrear(inv)
pd.concat({"disponibles": inv.groupby("evento").size(), "elegidos": seleccion.groupby("evento").size(),
           "elegidos_con_agua": seleccion[seleccion.frac_agua > C.FRAC_AGUA_POSITIVO].groupby("evento").size()}, axis=1).fillna(0).astype(int)

## 3. Extracción de parches
Cada parche `.npz` guarda `img` (VV_pre, VH_pre, VV_post, VH_post en dB, float16), `lab` (0 sin agua, 1 permanente, 2 inundación abierta, 3 urbana), `val` y `val_pre`.

In [ ]:
ruta_idx = C.SALIDA / "indice.csv"
if ruta_idx.exists() and not REHACER_PARCHES:
    indice = pd.read_csv(ruta_idx)
else:
    indice = D.extraer_parches(seleccion)
    indice.to_csv(ruta_idx, index=False)
indice.groupby(["particion", "dataset"]).agg(parches=("ruta", "size"), con_pre=("tiene_pre", "sum"),
                                             agua_media=("frac_agua", "mean"), inundacion_media=("frac_inundacion", "mean")).round(3)

## 4. Partición y estadísticas de normalización (solo Train)

In [ ]:
particion = {k: sorted(g.evento.unique().tolist()) for k, g in indice.groupby("particion")}
json.dump(particion, open(C.SALIDA / "particion.json", "w"), indent=2)
assert not set(particion["train"]) & set(particion["val"]), "un evento aparece en train y val"

ruta_stats = C.SALIDA / "stats.json"
if ruta_stats.exists() and not REHACER_PARCHES:
    stats = json.load(open(ruta_stats))
else:
    stats = D.estadisticas(indice)
    json.dump(stats, open(ruta_stats, "w"), indent=2)
pd.DataFrame(stats).set_index("canales").round(3)

## 5. Visualización de parches

In [ ]:
ejemplos = indice[(indice.frac_agua > 0.05) & indice.tiene_pre].groupby("dataset").head(2)
ejemplos = pd.concat([ejemplos, indice[indice.dataset == "colombia"].head(1)])
fig, ejes = plt.subplots(len(ejemplos), 5, figsize=(14, 2.9 * len(ejemplos)))
for fila, e in zip(np.atleast_2d(ejes), ejemplos.itertuples()):
    d = np.load(e.ruta)
    img = d["img"].astype("float32")
    for ax, x, t in zip(fila, [img[1], img[3], img[3] - img[1], d["lab"], d["val"]],
                        ["VH antes", "VH durante", "ΔVH", "etiqueta", "válido"]):
        cm, lo, hi = ("gray", -25, 0) if t.startswith("VH") else ("RdBu", -6, 6) if t == "ΔVH" else ("viridis", 0, 3)
        ax.imshow(x, cmap=cm, vmin=lo, vmax=hi)
        ax.set_title(f"{e.dataset}/{e.evento} · {t}", fontsize=7)
        ax.axis("off")
plt.tight_layout()

## 6. Entrenamiento
Para cada modo y cada encoder (ResNet34, VGG19, EfficientNet-B0, MobileNetV2): U-Net preentrenada en ImageNet, BCE + Dice sobre píxeles válidos, AdamW (lr 1e-4) con reducción en meseta, precisión mixta, lotes con la mitad de parches positivos, rotaciones de 90° y reflexiones, parada temprana por IoU de validación.

In [ ]:
resultados, historiales = [], []
for modo in C.MODOS:
    train, val = E.filtrar(indice, modo)
    print(f"\n== modo {modo}: train {len(train)} parches ({train.positivo.sum()} positivos) · "
          f"val {len(val)} ({val.positivo.sum()} positivos)")
    for enc in C.ENCODERS:
        ruta_pt = C.SALIDA / "modelos" / f"unet_{modo}_{enc}.pt"
        ruta_h = C.SALIDA / "modelos" / f"historial_{modo}_{enc}.csv"
        if ruta_pt.exists() and ruta_h.exists() and not REENTRENAR:
            h = pd.read_csv(ruta_h)
            mejor = h.loc[h.IoU.idxmax()].drop(["lr", "perdida_train"]).to_dict()
            print(f"{modo} {enc}: ya entrenado (IoU val {mejor['IoU']:.3f})")
        else:
            mejor, h = E.entrenar(enc, modo, train, val, stats, DISPOSITIVO)
            h.to_csv(ruta_h, index=False)
        resultados.append(mejor)
        historiales.append(h)
historial = pd.concat(historiales, ignore_index=True)
historial.to_csv(C.SALIDA / "historial.csv", index=False)

## 7. Comparación y selección del encoder (solo con validación internacional)

In [ ]:
resultados = pd.DataFrame(resultados).sort_values(["modo", "IoU"], ascending=[True, False]).reset_index(drop=True)
resultados.to_csv(C.SALIDA / "resultados_val.csv", index=False)
MEJOR = resultados.groupby("modo").first().encoder.to_dict()
print("Mejor encoder por modo:", MEJOR)
resultados.round(4)

In [ ]:
fig, ejes = plt.subplots(len(C.MODOS), 2, figsize=(11, 3.3 * len(C.MODOS)), squeeze=False)
for fila, (modo, gm) in zip(ejes, historial.groupby("modo")):
    for enc, g in gm.groupby("encoder"):
        fila[0].plot(g.epoca, g.perdida_train, marker="o", ms=3, label=enc)
        fila[1].plot(g.epoca, g.IoU, marker="o", ms=3, label=enc)
    fila[0].set_title(f"{modo} · pérdida train"); fila[1].set_title(f"{modo} · IoU val")
    for ax in fila:
        ax.set_xlabel("época"); ax.legend(fontsize=7)
plt.tight_layout()

## 8. Predicciones del mejor modelo en validación

In [ ]:
modelos = {m: I.cargar_modelo(m, MEJOR[m], DISPOSITIVO) for m in C.MODOS}
for modo, modelo in modelos.items():
    _, val = E.filtrar(indice, modo)
    ds = E.Parches(val, modo, stats, aumentar=False)
    muestras = val[val.positivo].groupby("evento").head(1).head(5)
    fig, ejes = plt.subplots(len(muestras), 3, figsize=(8, 2.7 * len(muestras)), squeeze=False)
    for fila, (k, e) in zip(ejes, muestras.iterrows()):
        x, y, v = ds[k]
        with torch.no_grad():
            p = torch.sigmoid(modelo(x[None].to(DISPOSITIVO)))[0, 0].cpu().numpy()
        for ax, z, t in zip(fila, [x[-1], y[0], (p >= C.UMBRAL) * v[0].numpy()], ["VH durante (norm.)", "referencia", "predicción"]):
            ax.imshow(z, cmap="gray" if t.startswith("VH") else "Blues"); ax.axis("off")
            ax.set_title(f"{modo} · {e.evento} · {t}", fontsize=7)
    plt.tight_layout(); plt.show()

## 9. Comparación adicional: Sen1Floods11 Bolivia (15 parches con etiqueta manual)
Solo modo `agua` (Bolivia no trae imagen previa). Referencia publicada: IoU de agua total ≈ 0.41 (FCNN, Bonafilia et al., 2020), con otro protocolo — la comparación es orientativa.

In [ ]:
filas, c_unet, c_otsu = [], np.zeros(4), np.zeros(4)
for r in sorted((C.BOLIVIA / "S1Hand").glob("*_S1Hand.tif")):
    post, perfil = I.leer_escena(r)
    ref = rasterio.open(C.BOLIVIA / "LabelHand" / r.name.replace("S1Hand", "LabelHand")).read(1)
    prob, valido = I.probabilidad(modelos["agua"], "agua", stats, post, None, DISPOSITIVO)
    m = valido & (ref != -1)
    c_unet += I.conteos(prob[m] >= C.UMBRAL, ref[m] == 1)
    c_otsu += I.conteos(post[1][m] < I.otsu(post[1][m]), ref[m] == 1)
pd.DataFrame([{"metodo": f"U-Net agua {MEJOR['agua']}", **E.metricas(*c_unet)},
              {"metodo": "Otsu VH por parche", **E.metricas(*c_otsu)}]).round(3)

## 10. Lima 2017 frente a EMSN038 (prueba externa)

**Advertencia importante.** Las capas usadas como referencia, `Potential_Maximum_flood` (atributo `source`: *river* / *maximun height*) y `Riverbank`, describen la **zona de inundación potencial máxima** y las riberas de los ríos Rímac y Chillón: no son una extensión de agua observada el 17/03/2017. La celda siguiente comprueba que la retrodispersión dentro de esas zonas casi no cambia entre el 05/03 y el 17/03. Por eso las métricas de esta sección miden la coincidencia con una zona de peligro, no el acierto del modelo.

In [ ]:
gdb = C.EMSN038
agua_ems = pd.concat([gpd.read_file(gdb, layer=c) for c in ["Potential_Maximum_flood", "Riverbank"]])
zona = gpd.read_file(gdb, layer="EMSN038_AOI").overlay(gpd.read_file(gdb, layer="Footprints"), how="intersection")
post17, perfil17 = I.leer_escena(C.LIMA / "lima_2017_0317.tif")
pre17 = I.leer_en_grilla(C.LIMA / "lima_2017_0305.tif", perfil17)
ref17 = I.rasterizar(agua_ems, perfil17).astype("int16")
ref17[I.rasterizar(zona, perfil17) == 0] = -1
v17 = D.validos_db(post17) & D.validos_db(pre17) & (ref17 != -1)

def pct(x): return np.percentile(x, [10, 50, 90]).round(1).tolist()
pd.DataFrame([{"zona": n, "píxeles": m.sum(), "VH antes p10/50/90": pct(pre17[1][m]),
               "VH durante p10/50/90": pct(post17[1][m]), "ΔVH p10/50/90": pct((post17[1] - pre17[1])[m])}
              for n, m in [("EMSN038 (agua potencial)", v17 & (ref17 == 1)), ("resto del área", v17 & (ref17 == 0))]])

In [ ]:
hand17 = I.a_grilla(C.HAND, perfil17)
bajo = hand17 < C.HAND_MAX
p_agua, _ = I.probabilidad(modelos["agua"], "agua", stats, post17, None, DISPOSITIVO)
p_cambio, _ = I.probabilidad(modelos["cambio"], "cambio", stats, post17, pre17, DISPOSITIVO)
u = I.otsu(post17[1][v17])
dvh = post17[1] - pre17[1]
capas17 = {f"U-Net agua {MEJOR['agua']}": p_agua >= C.UMBRAL,
           f"U-Net agua + HAND<{C.HAND_MAX}": (p_agua >= C.UMBRAL) & bajo,
           f"U-Net cambio {MEJOR['cambio']}": p_cambio >= C.UMBRAL,
           f"U-Net cambio + HAND<{C.HAND_MAX}": (p_cambio >= C.UMBRAL) & bajo,
           f"Otsu VH ({u:.1f} dB)": post17[1] < u,
           f"ΔVH < -3 dB + HAND<{C.HAND_MAX}": (dvh < -3) & bajo}
res17 = I.evaluar_mascaras(capas17, ref17, v17)
res17.round(3)

In [ ]:
I.mostrar([(np.ma.masked_where(~v17, post17[1]), "VH 17/03 (dB)", "gray", -25, 0),
           (np.ma.masked_where(~v17, dvh), "ΔVH (dB)", "RdBu", -6, 6),
           (np.ma.masked_where(~v17, p_agua), "prob. agua", "viridis", 0, 1),
           (np.ma.masked_where(~v17, p_cambio), "prob. cambio", "viridis", 0, 1),
           (np.ma.masked_where(ref17 == -1, ref17 == 1), "EMSN038 (potencial)", "Blues", 0, 1)], "Lima 2017")

## 11. Aplicación en Lima Sur 2023 (ciclón Yaku)
Imagen del 16/03/2023 (órbita 171 descendente) con la del 04/03/2023 como previa. Se exportan GeoTIFF de probabilidad y máscara para superponerlos en QGIS con los mapas CONIDA georreferenciados. La ocurrencia JRC > 80 % se marca como posible agua permanente (mar, lagunas) y no se cuenta como inundación.

In [ ]:
post23, perfil23 = I.leer_escena(C.LIMA / "lima_2023_0316.tif")
pre23 = I.leer_en_grilla(C.LIMA / "lima_2023_0304.tif", perfil23)
hand23, jrc23 = I.a_grilla(C.HAND, perfil23), I.a_grilla(C.JRC, perfil23)
permanente = np.nan_to_num(jrc23) > 80
p23_agua, v23_agua = I.probabilidad(modelos["agua"], "agua", stats, post23, None, DISPOSITIVO)
p23_cambio, v23 = I.probabilidad(modelos["cambio"], "cambio", stats, post23, pre23, DISPOSITIVO)
filtro = (hand23 < C.HAND_MAX) & ~permanente
inund23 = (p23_cambio >= C.UMBRAL) & filtro & v23
agua23 = (p23_agua >= C.UMBRAL) & filtro & v23_agua
oscuro_ambas = (post23[1] < -20) & (pre23[1] < -20) & v23

carpeta = C.SALIDA / "predicciones"
I.guardar_tif(carpeta / "lima_2023_0316_prob_agua.tif", np.where(v23_agua, p23_agua, -1), perfil23, "float32", -1)
I.guardar_tif(carpeta / "lima_2023_0316_prob_cambio.tif", np.where(v23, p23_cambio, -1), perfil23, "float32", -1)
I.guardar_tif(carpeta / "lima_2023_0316_inundacion.tif", np.where(v23, inund23, 255), perfil23, "uint8", 255)
I.guardar_tif(carpeta / "lima_2023_0316_agua.tif", np.where(v23_agua, agua23, 255), perfil23, "uint8", 255)
print(f"agua (modo agua): {100 * agua23[v23_agua].mean():.2f} % · inundación nueva (modo cambio): {100 * inund23[v23].mean():.2f} % · "
      f"oscuro en ambas fechas (arena/sombra): {100 * oscuro_ambas[v23].mean():.1f} %")
I.mostrar([(np.ma.masked_where(~v23, pre23[1]), "VH 04/03", "gray", -25, 0),
           (np.ma.masked_where(~v23, post23[1]), "VH 16/03", "gray", -25, 0),
           (np.ma.masked_where(~v23_agua, agua23), "agua (2 canales)", "Blues", 0, 1),
           (np.ma.masked_where(~v23, inund23), "inundación nueva (4 canales)", "Blues", 0, 1)], "Lima Sur 2023")

### 11.1 ¿En qué distritos? Área detectada por distrito (provincia de Lima)

In [ ]:
distritos = gpd.read_file(C.EMSN038, layer="Admin_MINAM_BAS_LIM_DISTRITOS")
por_distrito = I.resumen_por_distrito(inund23, v23, perfil23, distritos).merge(
    I.resumen_por_distrito(agua23, v23_agua, perfil23, distritos)[["distrito", "km2_detectados"]]
     .rename(columns={"km2_detectados": "km2_agua_2canales"}), on="distrito", how="left")
por_distrito.to_csv(C.SALIDA / "lima_2023_por_distrito.csv", index=False)
por_distrito.round(3).head(20)

## 12. Lectura de resultados y limitaciones

* Validación internacional alta + prueba en Lima baja = **cambio de dominio**, no necesariamente un mal modelo. Hay que reportarlo así.
* **EMSN038 no es verdad de campo** para el 17/03/2017 (zona potencial + riberas). Recomendación: presentarla como “coincidencia con zonas de peligro” o reemplazarla por polígonos digitalizados de eventos observados (CONIDA, INDECI/SINPAD, notas de prensa con coordenadas).
* **Desierto costero:** buena parte de Lima Sur aparece oscura en VH en *ambas* fechas (arena seca, sombras de radar en laderas). El modelo de 2 canales lo confunde con agua; el de 4 canales lo ve como “sin cambio”.
* **Escala temporal:** los anegamientos y huaicos de Lima Sur duran horas y dejan lodo (que no es oscuro en radar). Sentinel-1 pasa cada 12 días a las ~05:40 h; si la lluvia no coincide con la pasada, el agua ya no está.
* **Resolución:** 10 m es grueso para calles inundadas; la inundación urbana real produce doble rebote (más brillo), que solo se separa bien con coherencia SLC (UrbanSARFloods), no con GRD.
* Para **predecir** zonas propensas (no detectar), se necesitaría otro planteamiento: precipitación (SENAMHI, GPM IMERG, CHIRPS) + susceptibilidad (DEM, pendiente, HAND, quebradas) + inventario histórico de eventos (SIGRID/CENEPRED, INGEMMET) como etiqueta.